# **Homework #5: model deployment**

Step 1

In [0]:
df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(df_results)

In [0]:
spark.sql("USE default")

db_name = "default"

print("Using schema:", db_name)

Step 2

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE rf_predictions (
actual INT,
predicted INT
)
""")

spark.sql("""
CREATE OR REPLACE TABLE lr_predictions (
actual INT,
predicted INT
)
""")

In [0]:
from pyspark.sql.functions import when
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df_model = df_results.withColumn(
    "scored_points",
    when(df_results.points > 0, 1).otherwise(0)
).select(
    "grid", "driverId", "constructorId", "laps", "statusId", "scored_points"
).dropna()

df = df_model.toPandas()

X = df.drop("scored_points", axis=1)
y = df["scored_points"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

pred_df = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted": y_pred
})

rf_spark_df = spark.createDataFrame(pred_df)

rf_spark_df.write.mode("overwrite").saveAsTable("rf_predictions")

display(spark.table("rf_predictions"))

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt

mlflow.set_experiment("/Users/qt2167@columbia.edu/homework5_f1_model_deployment")

with mlflow.start_run(run_name="logistic_regression_model"):
    lr = LogisticRegression(max_iter=1000, random_state=42)

    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)

    acc = accuracy_score(y_test, y_pred_lr)
    prec = precision_score(y_test, y_pred_lr)
    rec = recall_score(y_test, y_pred_lr)
    f1 = f1_score(y_test, y_pred_lr)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(lr, name="logistic_regression_model")

    lr_pred_df = pd.DataFrame({
        "actual": y_test.reset_index(drop=True),
        "predicted": y_pred_lr
    })

    lr_pred_df.to_csv("/tmp/lr_predictions.csv", index=False)
    mlflow.log_artifact("/tmp/lr_predictions.csv")

    cm = confusion_matrix(y_test, y_pred_lr)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.savefig("/tmp/lr_confusion_matrix.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("/tmp/lr_confusion_matrix.png")

    print("accuracy:", acc)
    print("precision:", prec)
    print("recall:", rec)
    print("f1_score:", f1)

In [0]:
lr_spark_df = spark.createDataFrame(lr_pred_df)

lr_spark_df.write.mode("overwrite").saveAsTable("lr_predictions")

display(spark.table("lr_predictions"))

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

mlflow.set_experiment("/Users/qt2167@columbia.edu/homework5_f1_model_deployment")

with mlflow.start_run(run_name="random_forest_model"):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(rf, name="random_forest_model")

    pred_df.to_csv("/tmp/rf_predictions.csv", index=False)
    mlflow.log_artifact("/tmp/rf_predictions.csv")

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.savefig("/tmp/rf_confusion_matrix.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("/tmp/rf_confusion_matrix.png")

    print("accuracy:", acc)
    print("precision:", prec)
    print("recall:", rec)
    print("f1_score:", f1)

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.sql("SHOW TABLES").show()

spark.sql("SELECT COUNT(*) AS rf_count FROM rf_predictions").show()
spark.sql("SELECT COUNT(*) AS lr_count FROM lr_predictions").show()


This assignment builds two predictive models using the F1 results dataset. The two models are Random Forest and Logistic Regression.

For each model, I logged hyperparameters, the trained model object, four metrics, and two artifacts in MLflow. The four metrics are accuracy, precision, recall, and F1-score. The two artifacts are a prediction CSV file and a confusion matrix plot.

I created two prediction tables: `rf_predictions` for the Random Forest model and `lr_predictions` for the Logistic Regression model. Each table stores the actual label and the predicted label from the corresponding model.

Because I did not have permission to create a new schema in the workspace catalog, I used the default schema available to my account and created the two required prediction tables there.